In [1]:
'''
    Read in ERA5 level-based data, twice daily and construct a 'daily' average on pressure levels
'''

"\n    Read in ERA5 level-based data, twice daily and construct a 'daily' average on pressure levels\n"

In [2]:
import xarray as xr
import cfgrib
import numpy as np
import metpy.calc as mpcalc
from metpy.units import units

In [8]:
# Define the path to your GRIB file

print('--- BEGIN SCRIPT ---')

grib_dir  = '/glade/derecho/scratch/rneale/ERA5/download/'
grib_file = 'dtdt_param_1979_01_ytest_era5_modelevs.grib'

grib_all = grib_dir+grib_file

print(grib_file)

# Open the GRIB file using cfgrib and xarray
ds = xr.open_dataset(grib_all, engine='cfgrib')

# Display the dataset
print(ds)

# Extract model level data (e.g., temperature) and model level heights
temperature = ds['mttpm']  # Adjust variable name as needed
model_levels = ds['hybrid']  # Model level pressures in hPa

# Define target pressure levels for interpolation
target_pressures = np.array([1000, 925, 850, 700, 500, 300, 200, 100]) * units.hPa

# Interpolate the model level data to the target pressure levels
def interpolate_to_pressure_levels(model_levels, data, target_pressures):
    interpolated_data = []
    for time_index in range(data.shape[0]):
        interpolated_profile = []
        for lat_index in range(data.shape[1]):
            for lon_index in range(data.shape[2]):
                profile = data[time_index, :, lat_index, lon_index]
                pressure_profile = model_levels[:, lat_index, lon_index]
                interpolated_profile.append(mpcalc.interpolate_1d(target_pressures, pressure_profile, profile))
        interpolated_data.append(interpolated_profile)
    return np.array(interpolated_data)

# Perform interpolation
interpolated_temperature = interpolate_to_pressure_levels(model_levels, temperature, target_pressures)

# Convert the interpolated data back to an xarray DataArray
interp_temp_da = xr.DataArray(
    interpolated_temperature,
    coords={
        'time': temperature.time,
        'pressure': target_pressures.m,
        'latitude': temperature.latitude,
        'longitude': temperature.longitude
    },
    dims=['time', 'pressure', 'latitude', 'longitude']
)

# Create a new dataset with the interpolated data
interp_ds = xr.Dataset({'interpolated_temperature': interp_temp_da})

# Display the interpolated dataset
print(interp_ds)

--- BEGIN SCRIPT ---
dtdt_param_1979_01_ytest_era5_modelevs.grib
<xarray.Dataset> Size: 12GB
Dimensions:     (time: 62, hybrid: 88, values: 542080)
Coordinates:
  * time        (time) datetime64[ns] 496B 1979-01-01T06:00:00 ... 1979-01-31...
    step        timedelta64[ns] 8B ...
  * hybrid      (hybrid) float64 704B 50.0 51.0 52.0 53.0 ... 135.0 136.0 137.0
    latitude    (values) float64 4MB ...
    longitude   (values) float64 4MB ...
    valid_time  (time) datetime64[ns] 496B ...
Dimensions without coordinates: values
Data variables:
    mttpm       (time, hybrid, values) float32 12GB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2024-07-28T16:19 GRIB to CDM+CF via cfgrib-0.9.1...


IndexError: too many indices